# dim_shift

SCD1 shift-dimensie: per `shift_id` één rij — de laatste versie in `DWH_ORDER_HEADER` op `WA_FROMDATE` (inclusief `shift_start_time` en `shift_end_time` van die laatste rij). `MK_SHIFT` is de root-surrogaat `WKR_ORDER_HEADER` (stabiel over updates/deletes). `MA_ISDEL=1` als de laatste rij end-dated is. NULL `shift_id` (warn-rule) blijft als één Unknown-bucket bewaard via de PARTITION BY zelf.

Georkestreerd door `views/07_apply_views.ipynb`.

In [ ]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "DEMO";
USE CATALOG ${catalog};

CREATE OR REPLACE VIEW ${catalog}.DATAMART.DIM_SHIFT AS
WITH ranked AS (
  SELECT
    shift_id,
    shift_start_time,
    shift_end_time,
    WKR_ORDER_HEADER,
    WA_UNTODATE,
    ROW_NUMBER() OVER (PARTITION BY shift_id ORDER BY WA_FROMDATE DESC) AS rn,
    MIN(WA_CRUDDTS) OVER (PARTITION BY shift_id) AS MA_CREATEDATE,
    MAX(CASE WHEN WA_CRUD <> 'C' THEN WA_CRUDDTS END) OVER (PARTITION BY shift_id) AS MA_CHANGEDATE
  FROM ${catalog}.INTEGRATION.DWH_ORDER_HEADER
)
SELECT
  WKR_ORDER_HEADER AS MK_SHIFT,
  shift_id,
  shift_start_time,
  shift_end_time,
  MA_CREATEDATE,
  MA_CHANGEDATE,
  CASE WHEN WA_UNTODATE <> TIMESTAMP '9999-12-31 00:00:00' THEN 1 ELSE 0 END AS MA_ISDEL
FROM ranked
WHERE rn = 1;